<a href="https://colab.research.google.com/github/AdrianDVnqn/UA_MDM_Labo2_Grupo12/blob/LightGBM_Test/tutoriales/06_text_integrar_12_05_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
# prompt: Quisiera importar la libreria optuna

!pip install optuna
!pip install -U kaleido
!pip install optuna-dashboard
#!pip install kaleido


In [29]:
####IMPORTANTE CARGAR UTILS.PY
from google.colab import files
uploaded = files.upload()

Saving utils.py to utils (1).py


In [30]:
import numpy as np
import pandas as pd

from sklearn.metrics import cohen_kappa_score, accuracy_score,balanced_accuracy_score

from plotly import express as px

from utils import plot_confusion_matrix, get_artifact_filename

import os

from json import loads

from joblib import load, dump

import optuna
from optuna.artifacts import FileSystemArtifactStore, upload_artifact

In [31]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [32]:
# Definir la ruta base para tu Google Drive
BASE_DIR_DRIVE = '/content/drive/MyDrive'
PATH_TO_TRAIN = os.path.join(BASE_DIR_DRIVE, "Colab Notebooks/LABO_II/input/petfinder-adoption-prediction/train/train_final_thres.csv")
# Artefactos a subir a optuna
PATH_TO_TEMP_FILES = os.path.join(BASE_DIR_DRIVE, "Colab Notebooks/LABO_II/work/optuna_temp_artifacts")
# Artefactos que optuna gestiona
PATH_TO_OPTUNA_ARTIFACTS = os.path.join(BASE_DIR_DRIVE, "Colab Notebooks/LABO_II/work/optuna_artifacts")

In [33]:
# study_lgb = optuna.create_study(direction='maximize',
#                             storage="sqlite:///../work/db.sqlite3",  # Specify the storage URL here.
#                             study_name="04 - LGB Multiclass CV",
#                             load_if_exists = True)

# study_lgb = optuna.create_study(
#     direction='maximize',
#     storage="sqlite:///work/db.sqlite3",
#     study_name="04 - LGB Multiclass CV",
#     load_if_exists=True
# )

ruta_carpeta_work = '/content/drive/MyDrive/Colab Notebooks/LABO_II/work'

# Construye la ruta completa al archivo db.sqlite3
ruta_completa_db = os.path.join(ruta_carpeta_work, 'db_cv_12052025_1.sqlite3')

# Ahora utiliza esta ruta en la configuración de Optuna
storage = f"sqlite:///{ruta_completa_db}"

study_lgb = optuna.create_study(
    direction='maximize',
    storage=storage,
    study_name="04 - LGB Multiclass CV 12052025",
    load_if_exists=True
)



lgb_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study_lgb,'test')))

[I 2025-05-12 21:42:31,128] Using an existing study with name '04 - LGB Multiclass CV 12052025' instead of creating a new one.


In [34]:
lgb_dataset

,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,Color3,MaturitySize,...,stopwords_eliminadas,descripcion_para_analisis,Nombres_limpios,categoria_rescatista,cantidad_animales,disponibilidad_imagen,estado_sanitario,AgeCategory,State_importance,pred
10671,2,Elsa,2,265,0,2,1,4,7,2,...,"['is', 'is', 'this', 'is', 'for', 'me', 'to', ...","['elsa', 'female', 'kitten', 'age', 'months', ...",elsa,4,1,3,5,1,1,"[0.02322141249599711, 3.047856840081007, 1.442..."
9197,1,Gina,2,307,0,2,1,0,0,2,...,"['to', 'to', 'get', 'them', 'as', 'can', 'keep...","['pet', 'dog', 'gave', 'birth', 'puppies', 'lo...",gina,2,1,3,4,1,2,"[0.06568777179422862, 1.702299049664471, 1.152..."
7212,2,Bee,3,299,0,1,2,6,0,1,...,"['we', 'him', 'in', 'his', 'was', 'so', 'we', ...","['rescued', 'cat', 'saved', 'restaurant', 'leg...",bee,4,1,3,4,1,2,"[0.03364599028148262, 2.427540585708382, 1.487..."
10712,2,Mega,84,266,0,1,1,2,7,3,...,"['after', 'his', 'will', 'and', 'by', 'of', 'f...","['mega', 'named', 'strong', 'maggot', 'wound',...",mega,4,1,3,4,3,1,"[0.04714299718935783, 0.3655002929450484, 1.32..."
3681,2,NaN,2,265,266,3,1,2,7,1,...,['so'],"['hye', 'saya', 'ada', 'dua', 'ekor', 'anak', ...",NaN,1,2,3,5,1,3,"[0.13124458239716671, 1.9304072933891037, 1.20..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1015,1,Murni,24,5,307,2,3,5,0,2,...,"['is', 'but', 'can', 'be', 'around', 'she', 'h...","['murni', 'happy', 'doggie', 'shy', 'new', 'pe...",murni,4,1,3,1,2,1,"[0.0475220739067715, 0.4291818748300964, 0.982..."
9623,1,JoJo,72,83,0,1,1,0,0,2,...,"['was', 'by', 'he', 'was', 'an', 'and', 'he', ...","['jo', 'jo', 'black', 'cocker', 'abandoned', '...",jojo,4,1,3,1,3,1,"[0.047672862251968526, 1.3337219900503174, 1.2..."
8161,1,Little Blackie 1,1,307,0,1,1,0,0,2,...,['for'],"['puppy', 'adoption']",little blackie 1,4,1,3,5,1,1,"[1.1016573082387753, 0.8758756856520922, 1.850..."
6202,2,Baby,7,265,0,2,7,0,0,2,...,"['this', 'for', 'me', 'if', 'it', 'will', 'be'...","['hi', 'giving', 'cat', 'adoption', 'contact',...",baby,1,1,3,1,1,1,"[0.07487105849665703, 1.0594256145015262, 1.08..."


In [35]:
MODEL_NAME = '06 Bert_1'
MODEL_VERSION = '1.1'

# study_bert = optuna.create_study(direction='maximize',
#                             storage="sqlite:///work/db_bert.sqlite3",  # Specify the storage URL here.
#                             study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
#                             load_if_exists = True)

# Define la ruta a la carpeta 'work' en tu Google Drive
ruta_carpeta_work_bert = '/content/drive/MyDrive/Colab Notebooks/LABO_II/work'

# Construye la ruta completa al archivo db.sqlite3
ruta_completa_db_bert = os.path.join(ruta_carpeta_work_bert, 'db_bert_11052025.sqlite3')

# Ahora utiliza esta ruta en la configuración de Optuna
storage_bert = f"sqlite:///{ruta_completa_db_bert}"



study_bert = optuna.create_study(direction='maximize',
                                 storage=storage_bert,
                                 study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
                                 load_if_exists = True)

bert_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study_bert,'test')))

[I 2025-05-12 21:42:38,773] Using an existing study with name '06 Bert_1_1.1' instead of creating a new one.


In [36]:
bert_dataset

,PetID,pred,Type,Name,Age,Breed1,Breed2,Gender,Color1,Color2,...,stopwords_eliminadas,descripcion_para_analisis,Nombres_limpios,categoria_rescatista,cantidad_animales,disponibilidad_imagen,estado_sanitario,AgeCategory,State_importance,labels
0,23b64fe21,"[0.00063359767, 0.07574608, 0.05062803, 0.0056...",1,The Adorable Trio,2,307,307,1,1,0,...,"['these', 'was', 'from', 'in', 'we', 'to', 'ou...","['puppies', 'rescued', 'mechanic', 'shop', 'se...",the adorable trio,1,2,3,4,1,1,2
1,7983db8d9,"[1.653494e-05, 0.9782835, 0.008541423, 0.01291...",1,Lovebie,24,307,0,1,5,7,...,"['he', 'is', 'he', 'is', 'against', 'the', 'he...","['happy', 'dog', 'alert', 'strangers', 'approa...",lovebie,1,1,3,9,2,3,3
2,8d39e0fa1,"[2.1982429e-05, 0.008312243, 0.9903521, 0.0008...",1,NaN,2,307,307,2,1,2,...,"['they', 'are', 'and', 'found', 'this', 'by', ...","['adorable', 'quiet', 'obedient', 'cute', 'pup...",NaN,1,2,3,5,1,3,0
3,061378b30,"[0.0013142148, 0.010517364, 0.566219, 0.404417...",1,Murphy,3,31,307,1,1,0,...,"['for', 'to']","['lovely', 'puppy', 'looking', 'forever', 'hom...",murphy,4,1,3,9,1,1,2
4,2625dcb50,"[8.800309e-06, 3.4364424e-05, 0.00047203706, 0...",1,Jackie,8,307,0,2,1,0,...,"['was', 'to', 'at', 'where', 'she', 'is', 'for...","['jackie', 'brought', 'pets', 'wonderland', 'i...",jackie,2,1,3,1,1,3,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2393,fd48a560c,"[1.0327502e-05, 5.11318e-05, 0.0018107417, 0.9...",1,Wynna,2,307,0,2,1,2,...,"['was', 'found', 'in', 'please', 'give', 'her']","['bunny', 'abandoned', 'pj', 'old', 'town', 'h...",wynna,3,1,3,1,1,1,3
2394,79f358573,"[0.0003108269, 0.035154305, 0.02943502, 8.0158...",1,Scotty And Snowy,42,130,307,3,1,7,...,"['am', 'an', 'of', 'am', 'for', 'to', 'my', 't...","['owner', 'dogs', 'japanese', 'mixed', 'spitz'...",scotty and snowy,1,2,3,1,2,1,3
2395,1c901d8d7,"[0.0017749023, 0.0037819792, 0.003557747, 0.00...",1,NaN,1,307,0,3,1,2,...,"['the', 'at', 'one', 'of', 'the']","['somebody', 'left', 'puppies', 'charity', 'as...",NaN,1,3,2,5,1,3,4
2396,6984c9c58,"[2.5832927e-05, 0.00045419802, 0.99941397, 5.8...",2,Mr. Black,2,266,0,1,1,7,...,"['is', 'for', 'very', 'only', 'when', 'he', 'i...","['mr', 'black', 'playful', 'friendly', 'adorab...",mr black,3,3,3,5,1,1,2


In [37]:
merged_datasets = lgb_dataset[['PetID', 'pred', 'AdoptionSpeed']].rename({'pred':'lgb_pred_score'},axis=1).merge(bert_dataset[['PetID', 'pred']].rename({'pred':'bert_pred_score'},axis=1),
                  on='PetID', how='outer')



merged_datasets['bert_pred_score'] = [np.zeros(5) if type(i) is float else  i for i in merged_datasets['bert_pred_score'] ]

In [38]:
merged_datasets

,PetID,lgb_pred_score,AdoptionSpeed,bert_pred_score
0,0008c5398,"[0.1067783357618526, 0.836528217797271, 0.7339...",3,"[2.4032042e-05, 0.00010487018, 0.9989831, 0.00..."
1,0011d7c25,"[0.7400618853623159, 1.2368323504825802, 1.325...",2,"[0.00012656806, 0.00080582476, 0.0011982818, 0..."
2,002278114,"[0.02200946248748487, 0.4196726619439873, 1.67...",1,"[0.00023301592, 0.002945228, 0.9375273, 0.0587..."
3,0038234c6,"[0.11121563976495001, 1.500632826473763, 1.366...",4,"[0.0018242941, 0.023485912, 0.9604498, 0.00062..."
4,004709939,"[0.054417115030235524, 1.4405696952607903, 2.0...",1,"[7.933829e-05, 0.0006731945, 0.24005276, 0.003..."
...,...,...,...,...
2394,ff5158511,"[0.04727192412211468, 0.8536791406678489, 1.17...",2,"[2.2543823e-06, 3.520493e-05, 0.00018233924, 0..."
2395,ff706f8cc,"[0.024936099956474483, 1.3749867077589872, 1.7...",2,"[1.4887669e-05, 0.0004073571, 0.010546127, 0.0..."
2396,ff891d6b6,"[0.1264947420864747, 1.8966389859838797, 1.040...",3,"[7.477744e-05, 0.35079858, 0.03827971, 0.00457..."
2397,ff9d8cb25,"[0.20091806498384596, 1.2829625433552336, 2.06...",1,"[2.6026595e-05, 0.0043365336, 0.9748742, 0.020..."


In [39]:
merged_datasets['blend_pred_score'] = [r['lgb_pred_score']+r['bert_pred_score'] for i,r in merged_datasets.iterrows()]

In [40]:
merged_datasets['lgb_pred_score']

,lgb_pred_score
0,"[0.1067783357618526, 0.836528217797271, 0.7339..."
1,"[0.7400618853623159, 1.2368323504825802, 1.325..."
2,"[0.02200946248748487, 0.4196726619439873, 1.67..."
3,"[0.11121563976495001, 1.500632826473763, 1.366..."
4,"[0.054417115030235524, 1.4405696952607903, 2.0..."
...,...
2394,"[0.04727192412211468, 0.8536791406678489, 1.17..."
2395,"[0.024936099956474483, 1.3749867077589872, 1.7..."
2396,"[0.1264947420864747, 1.8966389859838797, 1.040..."
2397,"[0.20091806498384596, 1.2829625433552336, 2.06..."


In [41]:
merged_datasets['lgb_pred'] = [r.argmax() for r in merged_datasets['lgb_pred_score']]
merged_datasets['bert_pred'] = [r.argmax() for r in merged_datasets['bert_pred_score']]
merged_datasets['blended_pred'] = [r.argmax() for r in merged_datasets['blend_pred_score']]

In [42]:
merged_datasets['lgb_pred'] = [r.argmax() for r in merged_datasets['lgb_pred_score']]
merged_datasets['bert_pred'] = [r.argmax() for r in merged_datasets['bert_pred_score']]
merged_datasets['blended_pred'] = [r.argmax() for r in merged_datasets['blend_pred_score']]

In [43]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['lgb_pred'],
                    title = 'LGB Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                    merged_datasets['lgb_pred'],
                                                                    weights='quadratic')))

In [44]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['bert_pred'],
                    title = 'Bert Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                    merged_datasets['bert_pred'],
                                                                    weights='quadratic')))



In [45]:
plot_confusion_matrix(merged_datasets['AdoptionSpeed'],
                      merged_datasets['blended_pred'],
                    title = 'Blended Model Kappa: ' + str(cohen_kappa_score(merged_datasets['AdoptionSpeed'],
                                                                    merged_datasets['blended_pred'],
                                                                    weights='quadratic')))


In [46]:
MODEL_NAME = '04 ResNet_1005'
MODEL_VERSION = '1.0.1'

# study_resnet = optuna.create_study(direction='maximize',
#                             storage="sqlite:///../work/db.sqlite3",  # Specify the storage URL here.
#                             study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
#                             load_if_exists = True)

# Define la ruta a la carpeta 'work' en tu Google Drive
ruta_carpeta_work_resnet = '/content/drive/MyDrive/Colab Notebooks/LABO_II/work'
# Construye la ruta completa al archivo db.sqlite3
ruta_completa_db_resnet = os.path.join(ruta_carpeta_work_resnet, 'db_10052025.sqlite3')
# Ahora utiliza esta ruta en la configuración de Optuna
storage_resnet = f"sqlite:///{ruta_completa_db_resnet}"


study_resnet = optuna.create_study(
    direction='maximize',
    storage=storage_resnet,
    study_name=f'{MODEL_NAME}_{MODEL_VERSION}',
    load_if_exists=True
)


resnet_dataset = load(os.path.join(PATH_TO_OPTUNA_ARTIFACTS,get_artifact_filename(study_resnet,'test')))

[I 2025-05-12 21:43:26,674] Using an existing study with name '04 ResNet_1005_1.0.1' instead of creating a new one.


In [48]:
merged_datasets_lgb_bert = lgb_dataset.merge(bert_dataset, on='PetID', how='outer', suffixes=('_lgb', '_bert'))
merged_datasets_lgb_bert['bert_pred_score'] = [np.zeros(5) if type(i) is float else  i for i in merged_datasets['bert_pred_score'] ]

In [49]:
merged_datasets_lgb_bert = lgb_dataset[['PetID', 'pred', 'AdoptionSpeed']].rename({'pred':'lgb_pred_score'},axis=1).merge(bert_dataset[['PetID', 'pred']].rename({'pred':'bert_pred_score'},axis=1),
                  on='PetID', how='outer')

In [50]:
merged_datasets_final_final = merged_datasets_lgb_bert.merge(resnet_dataset[['PetID', 'pred']].rename({'pred':'resnet_pred_score'},axis=1),
                  on='PetID', how='outer')

In [51]:
merged_datasets_final_final['bert_pred_score'] = [np.zeros(5) if type(i) is float else  i for i in merged_datasets_final_final['bert_pred_score'] ]
merged_datasets_final_final['resnet_pred_score'] = [np.zeros(5) if type(i) is float else  i for i in merged_datasets_final_final['resnet_pred_score'] ]

In [52]:
merged_datasets_final_final

,PetID,lgb_pred_score,AdoptionSpeed,bert_pred_score,resnet_pred_score
0,0008c5398,"[0.1067783357618526, 0.836528217797271, 0.7339...",3,"[2.4032042e-05, 0.00010487018, 0.9989831, 0.00...","[-1.9458486, -0.0043535884, 1.2023433, 1.10391..."
1,0011d7c25,"[0.7400618853623159, 1.2368323504825802, 1.325...",2,"[0.00012656806, 0.00080582476, 0.0011982818, 0...","[-2.1744244, -0.2983307, 1.0597225, 0.62469965..."
2,002278114,"[0.02200946248748487, 0.4196726619439873, 1.67...",1,"[0.00023301592, 0.002945228, 0.9375273, 0.0587...","[-2.0341647, 0.84190595, 0.79452723, 0.3987306..."
3,0038234c6,"[0.11121563976495001, 1.500632826473763, 1.366...",4,"[0.0018242941, 0.023485912, 0.9604498, 0.00062...","[-2.6787448, 1.0057539, 1.159302, 0.6220832, -..."
4,004709939,"[0.054417115030235524, 1.4405696952607903, 2.0...",1,"[7.933829e-05, 0.0006731945, 0.24005276, 0.003...","[-1.6690625, 0.066984616, 0.37796998, 0.724677..."
...,...,...,...,...,...
2394,ff5158511,"[0.04727192412211468, 0.8536791406678489, 1.17...",2,"[2.2543823e-06, 3.520493e-05, 0.00018233924, 0...","[-1.4319173, 0.83807886, 1.2430862, 0.3130001,..."
2395,ff706f8cc,"[0.024936099956474483, 1.3749867077589872, 1.7...",2,"[1.4887669e-05, 0.0004073571, 0.010546127, 0.0...","[-1.6127261, 1.4383335, 1.3621353, 0.031383872..."
2396,ff891d6b6,"[0.1264947420864747, 1.8966389859838797, 1.040...",3,"[7.477744e-05, 0.35079858, 0.03827971, 0.00457...","[-2.137112, 0.40426302, 1.129489, 0.54325914, ..."
2397,ff9d8cb25,"[0.20091806498384596, 1.2829625433552336, 2.06...",1,"[2.6026595e-05, 0.0043365336, 0.9748742, 0.020...","[-1.7013699, 0.6615675, 0.500705, 0.1692328, 0..."


In [53]:
merged_datasets_final_final['blend_pred_score'] = [r['lgb_pred_score']+r['bert_pred_score']+r['resnet_pred_score'] for i,r in merged_datasets_final_final.iterrows()]

In [54]:
merged_datasets_final_final['lgb_pred'] = [r.argmax() for r in merged_datasets_final_final['lgb_pred_score']]
merged_datasets_final_final['bert_pred'] = [r.argmax() for r in merged_datasets_final_final['bert_pred_score']]
merged_datasets_final_final['resnet_pred'] = [r.argmax() for r in merged_datasets_final_final['resnet_pred_score']]
merged_datasets_final_final['blended_pred'] = [r.argmax() for r in merged_datasets_final_final['blend_pred_score']]


In [55]:
plot_confusion_matrix(merged_datasets_final_final['AdoptionSpeed'],
                      merged_datasets_final_final['blended_pred'],
                    title = 'Blended Model Kappa: ' + str(cohen_kappa_score(merged_datasets_final_final['AdoptionSpeed'],
                                                                    merged_datasets_final_final['blended_pred'],
                                                                    weights='quadratic')))